In [ ]:
using ArgParse
using Printf
using Dates
using JLD2
using ITensors
using ITensorMPS
using LinearAlgebra

const ROOT = normpath(joinpath(@__DIR__, ".."))
include(joinpath(ROOT, "QCSB", "QCSB.jl"))

include(joinpath(ROOT, "src", "circuit.jl"))
include(joinpath(ROOT, "src", "tci.jl"))

tci (generic function with 1 method)

In [952]:
siteinds(ψ2, 2)
# ITensors.state()

1-element Vector{Index{Vector{Pair{QN, Int64}}}}:
 (dim=2|id=691|"Qubit,Site,n=11") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1

In [965]:
foo = sum(ITensors.state(siteinds(ψ2)[1], "$i") for i in 0:dim(siteinds(ψ2)[1])-1)
Array(foo, inds(foo)...)

11-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [ ]:
function normalize!(state::DiagonalStateMPS)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)

    mats = Vector{ITensor}(undef, L-R+1)
    for j in R:L
        s = sites[j]
        bra = sum([dag(ITensors.state(s, ""))])
        mats[j-R+1] = dag(ITensors.state(s, "0") + ITensors.state(s, "1")) * ψ[j]
    end
end

ψ = deepcopy(state.mps);

R = 3

sites = siteinds(ψ)
L = length(sites)

if R > L
    return state, 0.0
end

if R > 1
    orthogonalize!(ψ, R-1)
end

mats = Vector{ITensor}(undef, L-R+1)
for j in R:L
    s = sites[j]
    mats[j-R+1] = dag(ITensors.state(s, "0") + ITensors.state(s, "1")) * ψ[j]
end

v₀ = mats[end]
logscale = 0.0
for j in 1:(L-R)
    v₁ = mats[end-j] * v₀
    n = norm(v₁)
    logscale += log(n)
    v₀ = v₁ / n
end

ψ[R-1] *= v₀

s = exp(+ logscale / (L - R + 1))
for j in 1:(L-R+1)
    ψ[j] *= s
end

ψ2 = MPS(ψ[1:R-1]);

In [915]:
function number_reduce(state::DiagonalStateMPS, A::Int)
    state = deepcopy(state)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)


    dist = ITensor(1.0)
    dist *= ψ[1]
    i1 = only(inds(ψ[1],"Site"))
    for (counter, site) in enumerate(1:A-1)
        i2 = only(inds(ψ[site+1],"Site"))
        i3 = siteind("Qudit", 1; dim=counter+2, conserve_qns=true)

        T = ITensor(dag(i1), dag(i2), i3)
        for a in 1:counter+1, b in 1:2
            c = a + b - 1
            T[i1 => a, i2 => b, i3 => c] = 1.0
        end

        dist *= ψ[site+1]*T
        i1 = i3
    end

    return MPS([dist, ψ[A+1:end]...])
end

number_reduce (generic function with 1 method)

In [944]:
L = 20
p = 0.1
T = Int(100/p)


state = neel_state(DiagonalStateMPS, L; conserve_number=true)
state = simple_circuit(state, L, T, p)

DiagonalStateMPS(MPS(20))

In [945]:
ψ2 = number_reduce(state, L÷2)


maxlinkdim(ψ2)

11

In [946]:
ψ2[1]

ITensor ord=2
(dim=11|id=743|"Link,l=10") <Out>
 1: QN("Number",10) => 1
 2: QN("Number",9) => 1
 3: QN("Number",8) => 1
 4: QN("Number",7) => 1
 5: QN("Number",6) => 1
 6: QN("Number",5) => 1
 7: QN("Number",4) => 1
 8: QN("Number",3) => 1
 9: QN("Number",2) => 1
 10: QN("Number",1) => 1
 11: QN("Number",0) => 1
(dim=11|id=950|"Qudit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 3: QN("Number",2) => 1
 4: QN("Number",3) => 1
 5: QN("Number",4) => 1
 6: QN("Number",5) => 1
 7: QN("Number",6) => 1
 8: QN("Number",7) => 1
 9: QN("Number",8) => 1
 10: QN("Number",9) => 1
 11: QN("Number",10) => 1
NDTensors.BlockSparse{ComplexF64, Vector{ComplexF64}, 2}

In [ ]:
measure(DiagonalStateMPS(ψ2), )

In [902]:
ψ = deepcopy(state.mps);

R = 3

sites = siteinds(ψ)
L = length(sites)

if R > L
    return state, 0.0
end

if R > 1
    orthogonalize!(ψ, R-1)
end

mats = Vector{ITensor}(undef, L-R+1)
for j in R:L
    s = sites[j]
    mats[j-R+1] = dag(ITensors.state(s, "0") + ITensors.state(s, "1")) * ψ[j]
end

v₀ = mats[end]
logscale = 0.0
for j in 1:(L-R)
    v₁ = mats[end-j] * v₀
    n = norm(v₁)
    logscale += log(n)
    v₀ = v₁ / n
end

ψ[R-1] *= v₀

s = exp(+ logscale / (L - R + 1))
for j in 1:(L-R+1)
    ψ[j] *= s
end

ψ2 = MPS(ψ[1:R-1]);

In [903]:
for i in 1:R-1
    ψ2 = orthogonalize!(ψ2, i)
    ψ2 = truncate(ψ2; cutoff=1e-8, maxdim=200)
end
maxlinkdim(ψ2)

4

In [875]:
ψ2 = truncate(ψ2; cutoff=1e-8, maxdim=200);

In [909]:
orthogonalize!(ψ2, 1)
maxlinkdim(ψ2)

4

In [911]:
ψ2[1]

ITensor ord=2
(dim=2|id=293|"Qubit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
(dim=4|id=111|"Link,l=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 3: QN("Number",0) => 1
 4: QN("Number",1) => 1
NDTensors.BlockSparse{ComplexF64, Vector{ComplexF64}, 2}

In [877]:
ψ2

5-element MPS:
 ((dim=2|id=293|"Qubit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1, (dim=50|id=576|"Link,l=1") <Out>
 1: QN("Number",0) => 5
 2: QN("Number",1) => 5
 3: QN("Number",2) => 5
 4: QN("Number",3) => 5
 5: QN("Number",4) => 5
 6: QN("Number",0) => 5
 7: QN("Number",1) => 5
 8: QN("Number",2) => 5
 9: QN("Number",3) => 5
 10: QN("Number",4) => 5)
 ((dim=2|id=138|"Qubit,Site,n=2") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1, (dim=84|id=209|"Link,l=2") <Out>
 1: QN("Number",0) => 4
 2: QN("Number",1) => 10
 3: QN("Number",2) => 10
 4: QN("Number",3) => 4
 5: QN("Number",0) => 4
 6: QN("Number",1) => 10
 7: QN("Number",2) => 10
 8: QN("Number",3) => 4
 9: QN("Number",0) => 4
 10: QN("Number",1) => 10
 11: QN("Number",2) => 10
 12: QN("Number",3) => 4, (dim=50|id=576|"Link,l=1") <In>
 1: QN("Number",0) => 5
 2: QN("Number",1) => 5
 3: QN("Number",2) => 5
 4: QN("Number",3) => 5
 5: QN("Number",4) => 5
 6: QN("Number",0) => 5
 7: QN("Number",1) => 5
 

In [ ]:
function traceright(state::DiagonalStateMPS, R::Int)
    ψ = copy(state.mps)
    sites = siteinds(ψ)
    L = length(sites)

    if R > L
        return state, 0.0
    end

    if R > 1
        orthogonalize!(ψ, R-1)
    end

    mats = Vector{ITensor}(undef, L-R+1)
    for j in R:L
        s = sites[j]
        mats[j-R+1] = dag(ITensors.state(s, "0") + ITensors.state(s, "1")) * ψ[j]
    end

    v₀ = mats[end]
    logscale = 0.0
    for j in 1:(L-R)
        v₁ = mats[end-j] * v₀
        n = norm(v₁)
        logscale += log(n)
        v₀ = v₁ / n
    end

    if R == 1
        return nothing, logscale + log(Complex(Array(v₀)[]))
    else
    
        ψ[R-1] *= v₀

        s = exp(+ logscale / (L - R + 1))
        for j in 1:(L-R+1)
            ψ[j] *= s
        end
        return DiagonalStateMPS(MPS(ψ[1:R-1])), logscale
    end
end

function normalize(state::DiagonalStateMPS)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)
    _, λ = traceright(state, 1)
    λ /= L
    
    s = exp(-λ)
    for j in 1:L
        ψ[j] *= s
    end
    return DiagonalStateMPS(ψ)
end

normalize (generic function with 1 method)

In [ ]:
function expval(state::DiagonalStateMPS, M::AbstractMatrix, pos::Int; cutoff=1E-8, maxdim=200, refs=0, conserve_qns=false)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites) - refs
    M_width = Int(log2(size(M)[1]))

    Mψ = apply(op(M, [sites[mod1(pos+i,L)] for i in 0:M_width-1]...), ψ; cutoff=cutoff, maxdim=maxdim)
    
    val = exp(traceright(DiagonalStateMPS(Mψ),1)[2] - traceright(DiagonalStateMPS(ψ),1)[2])
    return val
end

function measure(state::DiagonalStateMPS, M::AbstractMatrix, λ::Float64, pos::Int, m::Bool; cutoff=1E-8, maxdim=200, refs=0)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites) - refs
    M_width = Int(log2(size(M)[1]))

    Π = (I + (-1)^m * λ*M)/(sqrt(2*(1+λ^2)))
    g = op(Π*Π, [sites[mod1(pos+i,L)] for i in 0:M_width-1]...)

    ψ = apply(g, ψ; cutoff=cutoff, maxdim=maxdim)
    
    state = DiagonalStateMPS(ψ)
    truncate!(state; cutoff=cutoff, maxdim=maxdim)
    # println(maxlinkdim(state.mps))
    return normalize(DiagonalStateMPS(ψ))
end

function projectZ(state::DiagonalStateMPS, pos::Int, m::Bool; cutoff=1E-8, maxdim=200)
    ψ = state.mps
    s = siteind(ψ, pos)

    orthogonalize!(ψ, pos)

    which = m ? 2 : 1

    ψ[pos] *= dag(ITensors.state(s, which))
    ψ[pos] *= ITensors.state(s, m ? "1" : "0")

    truncate!(ψ; cutoff=cutoff, maxdim=maxdim)
    state = normalize(DiagonalStateMPS(ψ))

    return state
end

projectZ (generic function with 1 method)

In [859]:
L = 10
p = 0.1
T = Int(10/p)


state = neel_state(DiagonalStateMPS, L; conserve_number=true)
state = simple_circuit(state, L, T, p)

DiagonalStateMPS(MPS(10))

In [855]:
state2 = deepcopy(state)

state2 = traceright(state2, 6)[1]

DiagonalStateMPS(MPS(5))

In [858]:
orthogonalize!(state2.mps, 1)
maxlinkdim(state2.mps)

84

In [819]:
R = 6
ψ = copy(state.mps)
sites = siteinds(ψ)
L = length(sites)

if R > L
    return state, 0.0
end

mats = Vector{ITensor}(undef, L-R+1)
for j in R:L
    s = sites[j]
    mats[j-R+1] = dag(ITensors.state(s, "0") + ITensors.state(s, "1")) * ψ[j]
end

v₀ = mats[end]
logscale = 0.0
for j in 1:(L-R)
    v₁ = mats[end-j] * v₀
    n = norm(v₁)
    logscale += log(n)
    v₀ = v₁ / n
end

In [816]:
state2 = deepcopy(state)
# state2, _, _ = measure(state2, PauliZ, 1.0, 101:110; cutoff=1e-8, maxdim=200)
# state2 = traceright(state2, 200)[1]
# state2, _, _ = measure(state2, PauliZ, 1.0, 101:110; cutoff=1e-8, maxdim=200)
state2 = traceright(state2, 6)[1]
orthogonalize!(state2.mps, 3)
truncate!(state2; cutoff=1e-8, maxdim=200)

DiagonalStateMPS(MPS(5))

In [810]:
foo = reduce(*, state2.mps)

reshape(Array(foo, inds(foo)...), 32)

32-element Vector{ComplexF64}:
  0.003910432326542932 + 0.0im
  0.022622566341126794 + 0.0im
  0.022519936708588442 + 0.0im
   0.04182984086759172 + 0.0im
   0.02234830928047699 + 0.0im
    0.0424701163757443 + 0.0im
   0.04287371670002502 + 0.0im
  0.032997036819730735 + 0.0im
  0.022007955328131577 + 0.0im
  0.043453903035386904 + 0.0im
                       ⋮
  0.013207367702372807 + 0.0im
   0.04510247558559822 + 0.0im
   0.03971008207917583 + 0.0im
  0.040325747247089226 + 0.0im
  0.014351431459933346 + 0.0im
   0.04125795789480223 + 0.0im
  0.015015412497131748 + 0.0im
  0.015446280481861353 + 0.0im
 0.0017668474989433359 + 0.0im

In [817]:
orthogonalize!(state2.mps, 1)
orthogonalize!(state2.mps, 5)
orthogonalize!(state2.mps, 1)
orthogonalize!(state2.mps, 5)
orthogonalize!(state2.mps, 1)
orthogonalize!(state2.mps, 5)
maxlinkdim(state2.mps)

280

In [685]:
max(abs.(number_distribution_sample(state, L÷2, 10))...)

Time taken: 34.20971393585205 seconds for tracing, 1.390552043914795 seconds for distribution construction


0.6665723938654429

In [680]:
state2 = deepcopy(state)
B = 20

ms = rand(Bool, B)
for i in 1:B
    state2 = measure(state2, PauliZ, 1.0, 101+i-1, ms[i]; cutoff=1e-8, maxdim=200)
    # println(maxlinkdim(state2.mps))
end

In [730]:
orthogonalize!(state.mps, 150)
maxlinkdim(state.mps)

20

In [724]:
state3 = deepcopy(state2)

orthogonalize!(state3.mps, 101)
maxlinkdim(state3.mps)

# measure(state3, PauliZ, 1.0, 101:105; cutoff=1e-8, maxdim=200)

91

In [711]:
state2 = deepcopy(state)
A = 100
B = 5

ψ = state2.mps
sites = siteinds(ψ)
L = length(sites)

state2, _ = traceright(state2, A+B+1)
# truncate!(state2; cutoff=1e-8, maxdim=200)

# if B > 0
#     state2, _, _ = measure(state2, PauliZ, 1.0, A+1:A+B; cutoff=1e-8, maxdim=200)
#     # state2, _ = traceright(state2, A+1)
# end


# state2 = normalize(state2)

(DiagonalStateMPS(MPS(105)), 30.656750241032952)

In [681]:
state2 = deepcopy(state)
B = 10

ms = rand(Bool, B)
for i in 1:B
    state2 = projectZ(state2, 101+i-1, ms[i]; cutoff=1e-8, maxdim=200)
    # println(maxlinkdim(state2.mps))
end

In [682]:
function number_distribution_sample(state::DiagonalStateMPS, A::Int, B::Int)
    t1 = time()
    state = deepcopy(state)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)

    state, _ = traceright(state, A+B+1)
    if B > 0
        state, _, _ = measure(state, PauliZ, 1.0, A+1:A+B; cutoff=1e-8, maxdim=200)
        state, _ = traceright(state, A+1)
    end


    state = normalize(state)

    # return state
    ψ = state.mps

    t2 = time()
    dist = ITensor(1.0)
    dist *= ψ[1]
    i1 = only(inds(ψ[1],"Site"))
    for (counter, site) in enumerate(1:A-1)
        i2 = only(inds(ψ[site+1],"Site"))
        i3 = siteind("Qudit", 1; dim=counter+2, conserve_qns=true)

        T = ITensor(dag(i1), dag(i2), i3)
        for a in 1:counter+1, b in 1:2
            c = a + b - 1
            T[i1 => a, i2 => b, i3 => c] = 1.0
        end

        dist *= ψ[site+1]*T
        i1 = i3
    end

    t3 = time()

    println("Time taken: $(t2-t1) seconds for tracing, $(t3-t2) seconds for distribution construction")

    return Array(dist, inds(dist)...)
end

number_distribution_sample (generic function with 1 method)

In [512]:
function simple_circuit(state::DiagonalStateMPS, L::Int, T::Int, p::Float64; cutoff=1e-8, maxdim=200)
    SWAPn1 = decoherence_layer(state, SWAP, p, 1:2:L-1)
    SWAPn2 = decoherence_layer(state, SWAP, p, 2:2:L-1)

    for t in 1:T
        state = apply(SWAPn1, state; cutoff=cutoff, maxdim=maxdim)
        state = apply(SWAPn2, state; cutoff=cutoff, maxdim=maxdim)
        state = normalize(state)
        truncate!(state; cutoff=cutoff, maxdim=maxdim)
    end

    return state
end

simple_circuit (generic function with 1 method)

In [ ]:
L = 200
p = 0.1
T = Int(10/p)


state = neel_state(DiagonalStateMPS, L; conserve_number=true)
state = simple_circuit(state, L, T, p)


DiagonalStateMPS(MPS(200))

In [624]:
state2 = deepcopy(state)
B = 10

ms = rand(Bool, B)
for i in 1:B
    state2 = measure(state2, PauliZ, 1.0, 101+i-1, ms[i]; cutoff=1e-8, maxdim=200)
    println(maxlinkdim(state2.mps))
end

109
109
107
107
105
105
104
104
103
103
102
102
102
102
102
102
102
102
102
102


In [636]:
state2 = deepcopy(state)
B = 10

ms = rand(Bool, B)
for i in 1:B
    state2 = projectZ(state2, 101+i-1, ms[i]; cutoff=1e-8, maxdim=200)
    println(maxlinkdim(state2.mps))
end

140
267
481
641
735
808
863
903
934
955


In [628]:
truncate!(state2.mps; cutoff=1e-8, maxdim=200)

150-element MPS:
 ((dim=2|id=627|"Qubit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1, (dim=12|id=756|"Link,l=1") <Out>
 1: QN("Number",72) => 1
 2: QN("Number",73) => 1
 3: QN("Number",74) => 1
 4: QN("Number",75) => 1
 5: QN("Number",76) => 1
 6: QN("Number",77) => 1
 7: QN("Number",71) => 1
 8: QN("Number",72) => 1
 9: QN("Number",73) => 1
 10: QN("Number",74) => 1
 11: QN("Number",75) => 1
 12: QN("Number",76) => 1)
 ((dim=2|id=189|"Qubit,Site,n=2") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1, (dim=23|id=579|"Link,l=2") <Out>
 1: QN("Number",72) => 1
 2: QN("Number",73) => 1
 3: QN("Number",74) => 1
 4: QN("Number",75) => 1
 5: QN("Number",76) => 1
 6: QN("Number",77) => 1
 7: QN("Number",71) => 1
 8: QN("Number",72) => 2
 9: QN("Number",73) => 2
 10: QN("Number",74) => 2
 11: QN("Number",75) => 2
 12: QN("Number",76) => 2
 13: QN("Number",70) => 1
 14: QN("Number",71) => 1
 15: QN("Number",72) => 1
 16: QN("Number",73) => 1
 17: QN("Number",74) => 1
 1

In [599]:
samples = 1

state = traceright(state, 3L÷4+1)[1]

# number_distribution_sample(state, L÷2, 0)

B = 30

total = 0.0
for _ in 1:samples
    total += max(abs.(number_distribution_sample(state, L÷2, B))...)
end
total / samples

# ψ = number_distribution_sample(state, 2, 1).mps
# reshape(Array(ψ[1]*ψ[2], inds(ψ[1]*ψ[2])...), 4)

0.6362377711010386

In [987]:
data = jldopen("../send/optimal_decoder_2_L200_T20_gamma0p100_B10_samples1000_20260211_110740_4057923404.jld2")
data["prob1"]

0.8979261745933889

In [981]:
data["prob1"]

0.9867748164206331

In [976]:
using JLD2
data = jldopen("../output/optimal_decoder_2_L16_T10_gamma0p100_B4_samples5_20260211_021242_1019351799.jld2")

JLDFile /Users/jhauser/Code/U1_SWSSB/output/optimal_decoder_2_L16_T10_gamma0p100_B4_samples5_20260211_021242_1019351799.jld2 (read-only)
 ├─🔢 L
 ├─🔢 T
 ├─🔢 gamma
 ├─🔢 B
 ├─🔢 samples
 ├─🔢 prob1
 ├─🔢 prob2
 └─🔢 dt

In [979]:
data["prob1"]

0.852687034836966

In [978]:
data["prob2"]

0.7554947955956975

In [977]:
data["prob2"] - data["prob1"]^2

0.028419616216640353

In [517]:
L = 32

state = neel_state(DiagonalStateMPS, L; conserve_number=true)

expval(state, PauliZ, 1)

1.0 + 0.0im

In [513]:
traceright(DiagonalStateMPS(-state.mps), 1)

(nothing, 0.0 + 3.141592653589793im)

In [491]:
max(number_distribution_sample(state, L÷2, L÷4)...)

BoundsError: BoundsError: attempt to access 0-element Vector{Pair{QN, Int64}} at index [1]

In [475]:
traceright(state, 33)

(DiagonalStateMPS(MPS(32)), 0.0)

In [123]:
using ITensors

# scalar Z = ⟨+...+|ψ⟩, without building |+...+⟩ as an MPS
function overlap_plus(ψ::MPS)
    sites = siteinds(ψ)
    E = ITensor(1.0)
    for j in 1:length(sites)
        s = sites[j]

        # <+| = (⟨0|+⟨1|)/√2 or (⟨Up|+⟨Dn|)/√2 depending on siteset
        # We'll try common labels; adjust if needed.
        v0 = ITensors.state(s, "0")
        v1 = ITensors.state(s, "1")
        bra = dag(v0 + v1)

        E *= bra * ψ[j]
    end
    return scalar(E)
end

function normalize(state::DiagonalStateMPS)
    ψ = state.mps
    L = length(siteinds(ψ))

    Z = overlap_plus(ψ)               # ⟨+...+|ψ⟩
    λ = log(Z) / L                    # per-site log amplitude (use logabs if needed)
    s = exp(-λ)

    for j in 1:L
        ψ[j] *= s
    end
    return DiagonalStateMPS(ψ)
end


normalize (generic function with 1 method)

In [117]:
L = 10
state = neel_state(DiagonalStateMPS, L; conserve_number=true)
# state = normalize(state)

DiagonalStateMPS(MPS(10))

In [121]:
state.mps[4]

ITensor ord=3
(dim=1|id=911|"Link,l=3") <Out>
 1: QN("Number",1) => 1
(dim=2|id=35|"Qubit,Site,n=4") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
(dim=1|id=566|"Link,l=4") <In>
 1: QN("Number",2) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 3}

In [ ]:
# function normalize(state::DiagonalStateMPS)
#     ψ = state.mps
#     sites = siteinds(ψ)
#     L = length(sites)
#     λ = (loginner(MPS(sites, i -> "+"), ψ) + (L/2)*log(2)) / L
#     s = exp(-λ)
#     for j in 1:L
#         ψ[j] *= s
#     end
#     return DiagonalStateMPS(ψ)
# end

normalize (generic function with 1 method)

In [122]:
function simple_circuit(state::DiagonalStateMPS, L::Int, T::Int, p::Float64; cutoff=1e-8, maxdim=200)
    SWAPn1 = decoherence_layer(state, SWAP, p, 1:2:L-1)
    SWAPn2 = decoherence_layer(state, SWAP, p, 2:2:L-1)

    for t in 1:T
        state = apply(SWAPn1, state; cutoff=cutoff, maxdim=maxdim)
        state = apply(SWAPn2, state; cutoff=cutoff, maxdim=maxdim)
        state = normalize(state)
        # state /= norm(state.mps)
        truncate!(state; cutoff=cutoff, maxdim=maxdim)
    end

    return state
end

simple_circuit (generic function with 1 method)

In [83]:
L = 10
state = neel_state(DiagonalStateMPS, L; conserve_qns=true)

SWAPn1 = decoherence_layer(state, SWAP, 0.1, 1:2:L)
SWAPn2 = decoherence_layer(state, SWAP, 0.1, 2:3:L)

state = apply(SWAPn1, state)
state = apply(SWAPn2, state)
state /= norm(state.mps)
truncate!(state; cutoff=1e-8, maxdim=200)

state = apply(SWAPn1, state)
state = apply(SWAPn2, state)
truncate!(state; cutoff=1e-8, maxdim=200)

state = apply(SWAPn1, state)
state = apply(SWAPn2, state)
truncate!(state; cutoff=1e-8, maxdim=200)

DiagonalStateMPS(MPS(10))

In [141]:
"""
Build K with K|a⟩|b⟩ = |a+b⟩ (in the sense of adding the QNs of basis states).

Assumes:
- ind1, ind2 are QN Indices with 1-dimensional blocks (typical for "value=charge" encoding),
  OR at least that each basis state corresponds to a unique QN sector.
- output basis is labeled by total charge sectors, each dim 1, in increasing Q.
"""
function adder_itensor(ind1::Index, ind2::Index; outtags="Site,Out")
  d1, d2 = dim(ind1), dim(ind2)

  # # If not QN indices, fall back to dense.
  # if !(hasqns(ind1) && hasqns(ind2))
  #   dout = d1 + d2 - 1
  #   ind_out = Index(dout, outtags)
  #   K = ITensor(ind_out, ind1, ind2)
  #   for a in 1:d1, b in 1:d2
  #     c = a + b - 1
  #     K[ind_out => c, ind1 => a, ind2 => b] = 1.0
  #   end
  #   return K, ind_out
  # end

  # --- QN case ---
  # We assume each basis label a corresponds to a unique QN sector.
  # Get the QN of basis state a as the QN of the (unique) block containing it.
  # This helper is robust when each block has dim 1.
function qn_of_state(ind::Index, a::Int)
  offs = 0
  for b in 1:nblocks(ind)
    db = blockdim(ind, b)   # ← FIX
    if a <= offs + db
      return qn(ind, b)
    end
    offs += db
  end
  error("State $a out of range")
end

  # Build output blocks: all possible sums q1+q2, each with dim 1
  qset = Set{QN}()
  for a in 1:d1, b in 1:d2
    push!(qset, qn_of_state(ind1, a) + qn_of_state(ind2, b))
  end
  qlist = sort!(collect(qset))  # requires QN to be orderable; usually is
  outblocks = [q => 1 for q in qlist]
  ind_out = Index(outblocks, outtags)

  # Fill tensor. For each (a,b), pick the output sector with q = q(a)+q(b),
  # and map to its (unique) state (dim 1 sector → state index 1 within that block).
  K = ITensor(ind_out, ind1, ind2)

  # Map QN -> linear state label of ind_out (since each block dim 1)
  q_to_outstate = Dict{QN,Int}()
offs = 0
for bl in 1:nblocks(ind_out)
  q_to_outstate[qn(ind_out, bl)] = offs + 1
  offs += blockdim(ind_out, bl)   # ← FIX
end

  for a in 1:d1, b in 1:d2
    q = qn_of_state(ind1, a) + qn_of_state(ind2, b)
    c = q_to_outstate[q]
    K[ind_out => c, ind1 => a, ind2 => b] = 1.0
  end

  return K, ind_out
end

adder_itensor

In [149]:
Array(ψ[2], inds(ψ[2])...)

2×4×2 Array{Float64, 3}:
[:, :, 1] =
 0.596455  0.0       0.0        0.0
 0.0       0.801695  0.0390792  0.0

[:, :, 2] =
 0.0  0.965569  -0.0712417  0.0
 0.0  0.0        0.0        0.250203

In [ ]:
L = 4
state = neel_state(DiagonalStateMPS, L; conserve_number=true)

state = simple_circuit(state, L, L, 0.1)

DiagonalStateMPS(MPS(4))

In [188]:
ψ = state.mps

i1 = only(inds(ψ[1],"Site"))
i2 = only(inds(ψ[2],"Site"))

i3 = siteind("Qudit", 1; dim=3, conserve_qns=true)

T1 = ITensor(dag(i1), dag(i2), i3)
T1[i1 => 1, i2 => 1, i3 => 1] = 1.0
T1[i1 => 1, i2 => 2, i3 => 2] = 1.0
T1[i1 => 2, i2 => 1, i3 => 2] = 1.0
T1[i1 => 2, i2 => 2, i3 => 3] = 1.0

i4 = only(inds(ψ[3],"Site"))
i5 = siteind("Qudit", 1; dim=4, conserve_qns=true)

T2 = ITensor(dag(i3), dag(i4), i5)
T2[i3 => 1, i4 => 1, i5 => 1] = 1.0
T2[i3 => 1, i4 => 2, i5 => 2] = 1.0
T2[i3 => 2, i4 => 1, i5 => 2] = 1.0
T2[i3 => 2, i4 => 2, i5 => 3] = 1.0
T2[i3 => 3, i4 => 1, i5 => 3] = 1.0
T2[i3 => 3, i4 => 2, i5 => 4] = 1.0

i6 = only(inds(ψ[4],"Site"))
i7 = siteind("Qudit", 1; dim=5, conserve_qns=true)

T3 = ITensor(dag(i5), dag(i6), i7)
T3[i5 => 1, i6 => 1, i7 => 1] = 1.0
T3[i5 => 1, i6 => 2, i7 => 2] = 1.0
T3[i5 => 2, i6 => 1, i7 => 2] = 1.0
T3[i5 => 2, i6 => 2, i7 => 3] = 1.0
T3[i5 => 3, i6 => 1, i7 => 3] = 1.0
T3[i5 => 3, i6 => 2, i7 => 4] = 1.0
T3[i5 => 4, i6 => 1, i7 => 4] = 1.0
T3[i5 => 4, i6 => 2, i7 => 5] = 1.0


A = ((ψ[1] * ψ[2] * T1) * ψ[3] * T2) * ψ[4] * T3

ITensor ord=1
(dim=5|id=372|"Qudit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 3: QN("Number",2) => 1
 4: QN("Number",3) => 1
 5: QN("Number",4) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 1}

In [245]:
L = 4
state = neel_state(DiagonalStateMPS, L; conserve_number=true)

state = simple_circuit(state, L, 10L, 0.1)

DiagonalStateMPS(MPS(4))

In [207]:
state.mps[1]

ITensor ord=2
(dim=2|id=927|"Qubit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
(dim=2|id=998|"Link,l=1") <Out>
 1: QN("Number",20) => 1
 2: QN("Number",19) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 2}

In [ ]:
function traceright(state::DiagonalStateMPS, R::Int)
    ψ = copy(state.mps)
    sites = siteinds(ψ)
    L = length(sites)

    mats = Vector{ITensor}(undef, L-R+1)
    for j in R:L
        s = sites[j]
        mats[j-R+1] = dag(ITensors.state(s, "0") + ITensors.state(s, "1")) * ψ[j]
    end

    v₀ = mats[end]
    logscale = 0.0
    for j in 1:(L-R)
        v₁ = mats[end-j] * v₀
        n = norm(v₁)
        logscale += log(n)
        v₀ = v₁ / n
    end

    if R == 1
        return nothing, logscale
    else
    
        ψ[R-1] *= v₀

        s = exp(+ logscale / (L - R + 1))
        for j in 1:(L-R+1)
            ψ[j] *= s
        end
    return DiagonalStateMPS(MPS(ψ[1:R-1])), logscale
    end
end

function normalize(state::DiagonalStateMPS)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)
    _, λ = traceright(state, 1)
    λ /= L
    
    s = exp(-λ)
    for j in 1:L
        ψ[j] *= s
    end
    return DiagonalStateMPS(ψ)
end

normalize (generic function with 1 method)

In [309]:
L = 4
state = neel_state(DiagonalStateMPS, L; conserve_number=true)

state = simple_circuit(state, L, L, 0.1)

traceright(state, 1)

(nothing, 6.661338147750939e-16)

In [320]:
exp(traceright(DiagonalStateMPS(state.mps*4), 1)[2])

4.000000000000003

In [296]:
sum(diag(dense(state))

1.0 + 0.0im

In [257]:
R = 4

ψ = state.mps
sites = siteinds(ψ)
L = length(sites)

mats = Vector{ITensor}(undef, L-R+1)
for j in R:L
    s = sites[j]
    mats[j-R+1] = dag(ITensors.state(s, "0") + ITensors.state(s, "1")) * ψ[j]
end

v₀ = mats[end]
logscale = 0.0
for j in 1:(L-R)
    v₁ = mats[end-j] * v₀
    n = norm(v₁)
    logscale += log(n)
    v₀ = v₁ / n
end

ψ[R-1] * v₀

ITensor ord=2
(dim=2|id=635|"Qubit,Site,n=3") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
(dim=4|id=777|"Link,l=2") <In>
 1: QN("Number",2) => 1
 2: QN("Number",1) => 2
 3: QN("Number",0) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 2}

In [254]:
ψ = state.mps
mat = dag(ITensors.state(siteinds(ψ)[4], "0") + ITensors.state(siteinds(ψ)[4], "1")) * ψ[4]
ψ[3] * mat

ITensor ord=2
(dim=2|id=635|"Qubit,Site,n=3") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
(dim=4|id=777|"Link,l=2") <In>
 1: QN("Number",2) => 1
 2: QN("Number",1) => 2
 3: QN("Number",0) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 2}

In [250]:
mats

1-element Vector{ITensor}:
 ITensor ord=1
Dim 1: (dim=2|id=192|"Link,l=3") <In>
 1: QN("Number",1) => 1
 2: QN("Number",0) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 1}
 2-element
Block(2,)
 [2:2]
 1.0

Block(1,)
 [1:1]
 -1.0

In [249]:
ψ[3]

ITensor ord=3
(dim=2|id=635|"Qubit,Site,n=3") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
(dim=2|id=192|"Link,l=3") <Out>
 1: QN("Number",1) => 1
 2: QN("Number",0) => 1
(dim=4|id=777|"Link,l=2") <In>
 1: QN("Number",2) => 1
 2: QN("Number",1) => 2
 3: QN("Number",0) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 3}

In [244]:
ten = ψ[1]*ψ[2]*ψ[3]*ψ[4]
Array(ten, inds(ten)...)

2×2×2×2 Array{Float64, 4}:
[:, :, 1, 1] =
 0.0  0.0
 0.0  0.154702

[:, :, 2, 1] =
 0.0       0.166941
 0.161384  0.0

[:, :, 1, 2] =
 0.0       0.172097
 0.166941  0.0

[:, :, 2, 2] =
 0.177935  0.0
 0.0       0.0

In [230]:
Array(v₀, inds(v₀)...)

2-element Vector{Float64}:
 -1.0
  1.0

In [222]:
ψ = traceright(state, 40)

ErrorException: Attempting to contract IndexSet:

((dim=2|id=452|"Qubit,Site,n=39") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1, (dim=3|id=486|"Link,l=38") <In>
 1: QN("Number",2) => 1
 2: QN("Number",1) => 1
 3: QN("Number",0) => 1, (dim=2|id=733|"Link,l=39") <In>
 1: QN("Number",1) => 1
 2: QN("Number",0) => 1)

with IndexSet:

((dim=2|id=733|"Link,l=39") <In>
 1: QN("Number",1) => 1
 2: QN("Number",0) => 1,)

QN indices must have opposite direction to contract, but indices:

(dim=2|id=733|"Link,l=39") <In>
 1: QN("Number",1) => 1
 2: QN("Number",0) => 1

and:

(dim=2|id=733|"Link,l=39") <In>
 1: QN("Number",1) => 1
 2: QN("Number",0) => 1

do not have opposite directions.

In [ ]:
new_state

In [ ]:
function number_distribution_sample(state::DiagonalStateMPS, A::Int, B::Int)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)

    state = traceright(state, A+B+1)
    state, _, _ = measure(state, PauliZ, 1.0, A+1:A+B; cutoff=1e-8, maxdim=200)
    state = traceright(state, A+1)

    state = normalize(state)
    ψ = state.mps

    dist = ITensor()
    dist *= ψ[1]
    i1 = only(inds(ψ[1],"Site"))
    for (counter, site) in enumerate(A)
        i2 = only(inds(ψ[site+1],"Site"))
        i3 = siteind("Qudit", 1; dim=counter+2, conserve_qns=true)

        T = ITensor(dag(i1), dag(i2), i3)
        for a in 1:counter+1, b in 1:2
            c = a + b - 1
            T[i1 => a, i2 => b, i3 => c] = 1.0
        end

        dist *= ψ[2]*T
        i1 = i3
    end

    return Array(dist, inds(dist)...)
end

In [203]:
L = 4
state = neel_state(DiagonalStateMPS, L; conserve_number=true)

state = simple_circuit(state, L, 100L, 0.1)

ψ = state.mps

i1 = only(inds(ψ[1],"Site"))
i2 = only(inds(ψ[2],"Site"))

i3 = siteind("Qudit", 1; dim=3, conserve_qns=true)

T1 = ITensor(dag(i1), dag(i2), i3)
T1[i1 => 1, i2 => 1, i3 => 1] = 1.0
T1[i1 => 1, i2 => 2, i3 => 2] = 1.0
T1[i1 => 2, i2 => 1, i3 => 2] = 1.0
T1[i1 => 2, i2 => 2, i3 => 3] = 1.0

i4 = only(inds(ψ[3],"Site"))
i5 = siteind("Qudit", 1; dim=4, conserve_qns=true)

T2 = ITensor(dag(i3), dag(i4), i5)
T2[i3 => 1, i4 => 1, i5 => 1] = 1.0
T2[i3 => 1, i4 => 2, i5 => 2] = 1.0
T2[i3 => 2, i4 => 1, i5 => 2] = 1.0
T2[i3 => 2, i4 => 2, i5 => 3] = 1.0
T2[i3 => 3, i4 => 1, i5 => 3] = 1.0
T2[i3 => 3, i4 => 2, i5 => 4] = 1.0

s = only(inds(ψ[4], "Site"))
bra = dag(ITensors.state(s, "0") + ITensors.state(s, "1"))

B = ((ψ[1] * ψ[2] * T1) * ψ[3] * T2) * ψ[4] * bra
Array(B, inds(B)...)

4-element Vector{Float64}:
 0.0
 0.5000000000010888
 0.4999999999989111
 0.0

6

5-element Vector{Float64}:
 0.0
 0.0
 1.0000000000000002
 0.0
 0.0

In [176]:
i1 = siteind("Qubit", 1; conserve_number=true)
i2 = siteind("Qubit", 1; conserve_number=true)
i3 = siteind("Qudit", 1; dim=3, conserve_qns=true)

T = ITensor(dag(i1), dag(i2), i3)
T[i1 => 1, i2 => 1, i3 => 1] = 1.0
T[i1 => 1, i2 => 2, i3 => 2] = 1.0
T[i1 => 2, i2 => 1, i3 => 2] = 1.0
T[i1 => 2, i2 => 2, i3 => 3] = 1.0

1.0

In [172]:
T = ITensor(i, j)
T[i => 1, j => 1] = 1.0
T[i => 2, j => 2] = 1.0 
# T[i1 => 2, i2 => 1, i3 => 2] = 1.0
# T[i1 => 1, i2 => 2, i3 => 2] = 1.0
# T[i1 => 2, i2 => 2, k => 3] = 1.0

ErrorException: In `setindex!`, the element (2, 2) of ITensor: 
Dim 1: (dim=2|id=228|"Qudit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
Dim 2: (dim=3|id=780|"Qudit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 3: QN("Number",2) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 2}
 2×3
Block(1, 1)
 [1:1, 1:1]
 1.0
 you are trying to set is in a block with flux QN("Number",2), which is different from the flux QN("Number",0) of the other blocks of the ITensor. You may be trying to create an ITensor that does not have a well defined quantum number flux.

In [175]:
i = siteind("Qudit", 1; dim=2, conserve_number=true)
j = siteind("Qudit", 1; dim=3, conserve_qns=true)

T = ITensor(dag(i), j)
T[i => 1, j => 1] = 1.0
T[i => 2, j => 2] = 1.0 

1.0

In [173]:
using ITensors

# Domain: qubit with N=0,1
i2 = Index(QN("N",0)=>1, QN("N",1)=>1; tags="i2")

# Codomain: qutrit with N=0,1,2 (the N=2 state is "extraneous")
i3 = Index(QN("N",0)=>1, QN("N",1)=>1, QN("N",2)=>1; tags="i3")

# Embedding E: |0>↦|0>, |1>↦|1>, no amplitude into N=2
E = ITensor(dag(i3), i2)
E[i3=>1, i2=>1] = 1.0   # N=0 block
E[i3=>2, i2=>2] = 1.0   # N=1 block


1.0

In [174]:
E

ITensor ord=2
(dim=3|id=726|"i3") <In>
 1: QN("N",0) => 1
 2: QN("N",1) => 1
 3: QN("N",2) => 1
(dim=2|id=710|"i2") <Out>
 1: QN("N",0) => 1
 2: QN("N",1) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 2}

In [142]:
ψ = state.mps
adder1 = adder_itensor(only(inds(ψ[1],"Site")), only(inds(ψ[2],"Site")))

ErrorException: In `setindex!`, the element (2, 1, 2) of ITensor: 
Dim 1: (dim=3|id=348|"Out,Site") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 3: QN("Number",2) => 1
Dim 2: (dim=2|id=443|"Qubit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
Dim 3: (dim=2|id=41|"Qubit,Site,n=2") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 3}
 3×2×2
Block(1, 1, 1)
 [1:1, 1:1, 1:1]
[:, :, 1] =
 1.0
 you are trying to set is in a block with flux QN("Number",2), which is different from the flux QN("Number",0) of the other blocks of the ITensor. You may be trying to create an ITensor that does not have a well defined quantum number flux.

In [140]:
only(inds(ψ[1],"Site"))

(dim=2|id=443|"Qubit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1

In [125]:
L = 10
state = neel_state(DiagonalStateMPS, L; conserve_number=true)

state = simple_circuit(state, L, L, 0.1)

DiagonalStateMPS(MPS(10))

In [129]:
state.mps[1] * state.mps[2]

ITensor ord=3
(dim=2|id=443|"Qubit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
(dim=2|id=41|"Qubit,Site,n=2") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
(dim=4|id=273|"Link,l=2") <Out>
 1: QN("Number",5) => 1
 2: QN("Number",4) => 2
 3: QN("Number",3) => 1
NDTensors.BlockSparse{Float64, Vector{Float64}, 3}

In [103]:
measure(state, PauliZ, 1.0, 1)

ErrorException: Fluxes not all equal

In [107]:
ψ = state.mps
adder1 = adder_itensor(only(inds(ψ[1],"Site")), only(inds(ψ[2],"Site")))
# cum1 = adder1 * ψ[1] * ψ[2]

# adder2 = adder_itensor(only(inds(cum1,"Site")), only(inds(ψ[3],"Site")))
# cum2 = adder2 * cum1 * ψ[3]

MethodError: MethodError: no method matching adder_itensor(::Index{Vector{Pair{QN, Int64}}}, ::Index{Vector{Pair{QN, Int64}}})
The function `adder_itensor` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  adder_itensor(!Matched::Index{Int64}, !Matched::Index{Int64})
   @ Main ~/Code/U1_SWSSB/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W4sZmlsZQ==.jl:1


In [13]:
function adder_itensor(ind1::Index{Int}, ind2::Index{Int})
    d1 = dim(ind1)
    d2 = dim(ind2)
    dout = d1 + d2 - 1
    ind_out = Index(dout, "Site")
    K = ITensor(ind_out, ind1, ind2)
    for a in 1:d1
        for b in 1:d2
            c = a + b - 1
            K[ind_out => c, ind1 => a, ind2 => b] = 1.0
        end
    end
    return K
end

adder_itensor (generic function with 1 method)

In [ ]:
"""
Adder tensor K : H(d1) ⊗ H(d2) → H(dout)
with K|a⟩|b⟩ = |a+b⟩ (0-based labels).
"""
function adder_itensor(d1::Int, d2::Int; dout::Int=d1+d2-1,
                      tags1="s1", tags2="s2", tagsout="sout")
  @assert dout ≥ d1 + d2 - 1

  s1   = Index(d1, tags1)
  s2   = Index(d2, tags2)
  sout = Index(dout, tagsout)

  # K has one ket index (sout) and two bra indices (dag(s1), dag(s2))
  K = ITensor(sout, dag(s1), dag(s2))

  # Fill: K[sout=c, s1=a, s2=b] = 1 if c = a+b
  for a in 0:d1-1
    for b in 0:d2-1
      c = a + b
      K[sout => c+1, s1 => a+1, s2 => b+1] = 1.0
    end
  end

  return K, s1, s2, sout
end

K, s1, s2, sout = adder_itensor(3, 5)  # dout defaults to d1+d2-1 = 7
